In [99]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("Data")
RAW_DIR = DATA_DIR / "Raw"
PROCESSED_DIR = DATA_DIR / "Processed"
for directory in (RAW_DIR, PROCESSED_DIR):
    directory.mkdir(parents=True, exist_ok=True)


def load_csv_datasets(raw_dir):
    csv_paths = sorted(raw_dir.glob("*.csv"))
    datasets = {path.stem: pd.read_csv(path) for path in csv_paths}
    return datasets, csv_paths


raw_csv_datasets, raw_csv_paths = load_csv_datasets(RAW_DIR)
expected_csv_stems = {path.stem for path in raw_csv_paths}
loaded_csv_stems = set(raw_csv_datasets)
missing_csv_stems = sorted(expected_csv_stems - loaded_csv_stems)
if missing_csv_stems:
    raise RuntimeError(f"Missing CSV loads for: {', '.join(missing_csv_stems)}")

governance_workbook_path = RAW_DIR / "Worldwide_Governance_Indicators.xlsx"
if not governance_workbook_path.exists():
    raise FileNotFoundError(f"Missing workbook: {governance_workbook_path}")

dfx = pd.read_excel(governance_workbook_path)

# Legacy aliases used by later cells
# Keep these names stable so the rest of the notebook does not need to change.
df_rd = raw_csv_datasets["OECD_RD_GDP_2000_2025"]
df3 = raw_csv_datasets["OECD_Productivity_Variation_2000_2025"]
df4 = raw_csv_datasets["OECD_Tertiary_Education_2000_2025"]
df5 = raw_csv_datasets["OECD_Unemployment_Rate_2000_2025"]
df6 = raw_csv_datasets["OECD_Inflation_CPI_2000_2025"]
dft = raw_csv_datasets.get("OECD_Trust_Institutions_2000_2025")
dfe = raw_csv_datasets.get("Eurostat_Energy_Import_Dependency_2000_2025", raw_csv_datasets["Global_Energy_Dependency_0_100"] )
dfg = raw_csv_datasets["OECD_GDP_Growth_2000_2025"]
df_ge = raw_csv_datasets["Global_Energy_Dependency_0_100"]
df_eur = raw_csv_datasets["Eurostat_Public_Debt_GDP_2000_2025"]
df_gd = raw_csv_datasets["Global_Public_Debt_GDP_2000_2025"]

print("Loaded CSV datasets:")
print(", ".join(sorted(raw_csv_datasets)))
print("Loaded workbook:")
print(governance_workbook_path.name)
print("Inventory ready in memory")

Loaded CSV datasets:
Eurostat_Public_Debt_GDP_2000_2025, Global_Energy_Dependency_0_100, Global_Public_Debt_GDP_2000_2025, OECD_GDP_Growth_2000_2025, OECD_Inflation_CPI_2000_2025, OECD_Productivity_Variation_2000_2025, OECD_RD_GDP_2000_2025, OECD_Tertiary_Education_2000_2025, OECD_Unemployment_Rate_2000_2025
Loaded workbook:
Worldwide_Governance_Indicators.xlsx
Inventory ready in memory


In [ ]:
import pandas as pd
import pycountry

def iso2_to_iso3(code):
    if pd.isna(code):
        return None
    code = str(code).strip().upper()
    if not code:
        return None
    country = pycountry.countries.get(alpha_2=code)
    if country is None:
        try:
            country = pycountry.countries.lookup(code)
        except LookupError:
            return None
    return country.alpha_3


def clean_dataset(df, fix_iso2=False):
    df = df.copy()
    df.columns = df.columns.str.lower().str.strip()

    required_columns = {"country", "year", "value"}
    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(f"Missing required columns: {', '.join(sorted(missing_columns))}")

    df = df[["country", "year", "value"]]
    df["country"] = df["country"].astype("string").str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    if fix_iso2:
        df["country"] = df["country"].apply(iso2_to_iso3)

    df["country"] = df["country"].fillna("MISSING_COUNTRY")
    df = df.dropna(subset=["year"] )
    df["year"] = df["year"].astype(int)
    df = df.drop_duplicates().sort_values(["country", "year"]).reset_index(drop=True)
    df["value"] = df.groupby("country", dropna=False)["value"].transform(
        lambda x: x.interpolate(limit_direction="both").ffill().bfill()
    )
    return df

csv_dataset_names = sorted(raw_csv_datasets.keys())
files = [f"{dataset_name}.csv" for dataset_name in csv_dataset_names]
datasets = {}
for file in files:
    dataset_key = file.replace(".csv", "")
    df = raw_csv_datasets[dataset_key].copy()
    if "Eurostat" in file:
        datasets[dataset_key] = clean_dataset(df, fix_iso2=True)
    else:
        datasets[dataset_key] = clean_dataset(df, fix_iso2=False)

pivot_datasets = {}
for name, df in datasets.items():
    pivot_datasets[name] = df.pivot_table(
        index="year",
        columns="country",
        values="value",
        aggfunc="mean"
    ).sort_index()

dfs_long = []
for name, df in pivot_datasets.items():
    temp = df.reset_index().melt(
        id_vars="year",
        var_name="country",
        value_name="value"
    )
    temp["indicator"] = name
    dfs_long.append(temp)

df_all = pd.concat(dfs_long, ignore_index=True)
final_df = df_all.pivot_table(
    index=["year", "country"],
    columns="indicator",
    values="value"
).reset_index()

final_clean = final_df.copy()
numeric_cols = final_clean.select_dtypes(include="number").columns
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.interpolate(limit_direction="both")
 )
final_clean[numeric_cols] = final_clean.groupby("country")[numeric_cols].transform(
    lambda x: x.fillna(x.mean())
)

def country_to_iso3(country_name):
    if pd.isna(country_name):
        return None
    country_name = str(country_name).strip()
    if not country_name:
        return None
    try:
        return pycountry.countries.lookup(country_name).alpha_3
    except LookupError:
        return None

governance_df = pd.read_excel(RAW_DIR / "Worldwide_Governance_Indicators.xlsx")
governance_df = governance_df.loc[governance_df["Year"] >= 2000].copy()
governance_df["Economy (name)"] = governance_df["Economy (name)"].apply(country_to_iso3)

drop_columns = [
    "ID variable (economy code/ gov. dimension/ year)",
    "Economy (code)",
    "Region",
    "Income classification",
    "Governance dimension",
    "Number of sources",
    "Governance estimate (approx. -2.5 to +2.5)",
    "Standard error (estimate)",
    "Lower threshold (90% conf. int. estimate)",
    "Upper threshold (90% conf. int. estimate)",
    "Standard error (gov. score)",
    "Lower threshold (90% conf. int. score)",
    "Upper threshold (90% conf. int. score)",
    "ADB mean",
    "AFR mean",
    "ARB mean",
    "ASB mean",
    "ASD mean",
    "BPS mean",
    "BTI mean",
    "CCR mean",
    "EBR mean",
    "EIU mean",
    "EQI mean",
    "EUB mean",
    "FRH mean",
    "GCB mean",
    "GCS mean",
    "GII mean",
    "GWP mean",
    "HER mean",
    "HRM mean",
    "HUM mean",
    "IFD mean",
    "IJT mean",
    "IRP mean",
    "LBO mean",
    "MSI mean",
    "OBI mean",
    "PIA mean",
    "PRS mean",
    "RSF mean",
    "VAB mean",
    "VDM mean",
    "WBS mean",
    "WCY mean",
    "WJP mean",
    "WMO mean"
]
governance_df = governance_df.drop([column for column in drop_columns if column in governance_df.columns], axis=1)
governance_df = governance_df.rename(columns={"Economy (name)": "country", "Year": "year", "Governance score (0-100)": "Worldwide_Governance_Indicators"})

final_merged = pd.merge(final_clean, governance_df, on=["country", "year"], how="inner")

final_dataset_all_12_columns = final_merged[[
    "year",
    "country",
    "Eurostat_Public_Debt_GDP_2000_2025",
    "Global_Energy_Dependency_0_100",
    "Global_Public_Debt_GDP_2000_2025",
    "OECD_GDP_Growth_2000_2025",
    "OECD_Inflation_CPI_2000_2025",
    "OECD_Productivity_Variation_2000_2025",
    "OECD_RD_GDP_2000_2025",
    "OECD_Tertiary_Education_2000_2025",
    "OECD_Unemployment_Rate_2000_2025",
    "Worldwide_Governance_Indicators",
]].copy()

final_dataset_ordinal_part = final_merged[[
    "year",
    "country",
    "OECD_RD_GDP_2000_2025",
    "Worldwide_Governance_Indicators",
    "Global_Public_Debt_GDP_2000_2025",
    "Global_Energy_Dependency_0_100",
    "OECD_Tertiary_Education_2000_2025",
]].copy()

final_dataset_validation_set = final_merged[[
    "year",
    "country",
    "OECD_Inflation_CPI_2000_2025",
    "OECD_Unemployment_Rate_2000_2025",
    "Worldwide_Governance_Indicators",
    "OECD_Productivity_Variation_2000_2025",
    "OECD_GDP_Growth_2000_2025",
]].copy()

final_test_missing_rows = final_dataset_all_12_columns[final_dataset_all_12_columns.isna().any(axis=1)].copy()
final_test_missing_values = (
    final_dataset_all_12_columns
    .assign(country=final_dataset_all_12_columns["country"].fillna("MISSING_COUNTRY"))
    .melt(id_vars=["country"], var_name="variable", value_name="observed_value")
    .assign(is_missing=lambda frame: frame["observed_value"].isna())
    .groupby(["country", "variable"], as_index=False)["is_missing"]
    .sum()
    .rename(columns={"is_missing": "missing_count"})
    .sort_values(["country", "variable"])
    .reset_index(drop=True)
 )

column_missing_counts = final_dataset_all_12_columns.isna().sum()
columns_with_no_more_than_three_missing = column_missing_counts[column_missing_counts <= 3].index.tolist()
final_dataset_max_3_missing_columns = final_dataset_all_12_columns[columns_with_no_more_than_three_missing].copy()

final_dataset_all_12_columns.to_csv(PROCESSED_DIR / "final_dataset_all_12_columns.csv", index=False)
final_dataset_ordinal_part.to_csv(PROCESSED_DIR / "final_dataset_ordinal_part.csv", index=False)
final_dataset_validation_set.to_csv(PROCESSED_DIR / "final_dataset_validation_set.csv", index=False)
final_test_missing_values.to_csv(PROCESSED_DIR / "final_test_missing_values.csv", index=False)
final_test_missing_rows.to_csv(PROCESSED_DIR / "final_test_missing_rows.csv", index=False)
final_dataset_max_3_missing_columns.to_csv(PROCESSED_DIR / "final_dataset_max_3_missing_columns.csv", index=False)

print("Final datasets exported:")
print("- final_dataset_all_12_columns.csv")
print("- final_dataset_ordinal_part.csv")
print("- final_dataset_validation_set.csv")
print("- final_test_missing_values.csv")
print("- final_test_missing_rows.csv")
print("- final_dataset_max_3_missing_columns.csv")
print(f"Columns kept in max-3-missing dataset: {len(columns_with_no_more_than_three_missing)}")

Final datasets exported:
- final_dataset_all_12_columns.csv
- final_dataset_ordinal_part.csv
- final_dataset_validation_set.csv
- final_test_missing_values.csv
- final_test_missing_rows.csv
- final_dataset_max_3_missing_columns.csv
Columns kept in max-3-missing dataset: 4
